# 0b. Model Explorer: BE & SC

Interactive exploration of how BE and SC model parameters affect
behaviour. Build intuition before any fitting or analysis.

**Covers:**
1. Interactive parameter sliders (BE and SC)
2. Direct BE vs SC comparison at matched parameters
3. Phase presets (naive / expert / post-shift)

Parameter sweeps and shift predictions live in 1b and 6a respectively.

In [ ]:
from shared_setup import *

from ipywidgets import interact, FloatSlider

from models.BE_core import BEParams, BEState, BEModel
from models.SC_core import SCParams, SCState, SCModel
from analysis.stimulus_distribution import sample_distribution

## 1. Setup

In [ ]:
N_TRIALS = 400
BURN_IN = 1000
SEED = 42
N_BINS = 8

rng_stim = np.random.default_rng(SEED)
stimuli = rng_stim.uniform(-1, 1, N_TRIALS)
categories = (stimuli > 0).astype(int)

def simulate_be(sigma_percep, A_repulsion, eta_learning, eta_relax, seed=SEED):
    params = BEParams(sigma_percep=sigma_percep, A_repulsion=A_repulsion,
                      eta_learning=eta_learning, eta_relax=eta_relax)
    rng = np.random.default_rng(seed)
    state = BEModel.run_burn_in(params, BEState.initial_uniform(), BURN_IN, seed)
    choices, p_B, _, _ = BEModel.simulate_session(
        params, state, stimuli, categories, rng)
    um, _, info = compute_update_matrix(stimuli, choices, categories, n_bins=N_BINS)
    acc = float(np.mean(choices == categories))
    return um, choices, acc, info


def simulate_sc(sigma_percep, A_repulsion, gamma, sigma_update, seed=SEED):
    params = SCParams(sigma_percep=sigma_percep, A_repulsion=A_repulsion,
                      gamma=gamma, sigma_update=sigma_update)
    rng = np.random.default_rng(seed)
    state = SCModel.create_initial_state(params=params, burn_in=BURN_IN, seed=seed)
    choices, p_B, _, _ = SCModel.simulate_session(
        params, state, stimuli, categories, rng)
    um, _, info = compute_update_matrix(stimuli, choices, categories, n_bins=N_BINS)
    acc = float(np.mean(choices == categories))
    return um, choices, acc, info


def plot_model_output(um, choices, acc, title=''):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    # Psychometric
    valid = ~np.isnan(choices)
    plot_psychometric(stimuli[valid], choices[valid], ax=axes[0])
    axes[0].set_title(f'Psychometric (acc={acc:.2f})')
    # Update matrix
    plot_update_matrix(um, ax=axes[1])
    axes[1].set_title('Update matrix')
    # Serial dependence profile
    midpoints = np.linspace(-1, 1, N_BINS + 1)
    midpoints = (midpoints[:-1] + midpoints[1:]) / 2
    axes[2].plot(midpoints, np.nanmean(um, axis=1), 'o-', color='steelblue')
    axes[2].axhline(0, color='k', lw=0.5)
    axes[2].set_xlabel('Previous stimulus bin')
    axes[2].set_ylabel('Mean Δ P(B)')
    axes[2].set_title('SD profile')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 2. Interactive Explorers

### 2.1 BE Model

In [ ]:
def be_interactive(sigma_percep, A_repulsion, eta_learning, eta_relax):
    um, choices, acc, info = simulate_be(
        sigma_percep, A_repulsion, eta_learning, eta_relax)
    plot_model_output(um, choices, acc, title='BE Model')

interact(be_interactive,
         sigma_percep=FloatSlider(min=0.05, max=0.5, step=0.05, value=0.15),
         A_repulsion=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.1),
         eta_learning=FloatSlider(min=0.05, max=0.9, step=0.05, value=0.35),
         eta_relax=FloatSlider(min=0.01, max=0.4, step=0.02, value=0.12))

### 2.2 SC Model

In [ ]:
def sc_interactive(sigma_percep, A_repulsion, gamma, sigma_update):
    um, choices, acc, info = simulate_sc(
        sigma_percep, A_repulsion, gamma, sigma_update)
    plot_model_output(um, choices, acc, title='SC Model')

interact(sc_interactive,
         sigma_percep=FloatSlider(min=0.05, max=0.5, step=0.05, value=0.15),
         A_repulsion=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.1),
         gamma=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.95),
         sigma_update=FloatSlider(min=0.1, max=1.0, step=0.1, value=0.3))

## 3. BE vs SC Direct Comparison

Same perceptual parameters, comparable effective learning rates.

In [ ]:
SHARED = dict(sigma_percep=0.15, A_repulsion=0.10)
BE_PARAMS = dict(**SHARED, eta_learning=0.35, eta_relax=0.12)
SC_PARAMS = dict(**SHARED, gamma=0.95, sigma_update=0.3)

be_um, be_ch, be_acc, _ = simulate_be(**BE_PARAMS)
sc_um, sc_ch, sc_acc, _ = simulate_sc(**SC_PARAMS)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
plot_update_matrix(be_um, ax=axes[0], title=f'BE (acc={be_acc:.2f})')
plot_update_matrix(sc_um, ax=axes[1], title=f'SC (acc={sc_acc:.2f})')
diff = be_um - sc_um
vmax = np.nanmax(np.abs(diff))
plot_update_matrix(diff, ax=axes[2], title='BE − SC', vmin=-vmax, vmax=vmax)

# SD profiles
midpoints = np.linspace(-1, 1, N_BINS + 1)
midpoints = (midpoints[:-1] + midpoints[1:]) / 2
axes[3].plot(midpoints, np.nanmean(be_um, axis=1), 'o-', label='BE', color='steelblue')
axes[3].plot(midpoints, np.nanmean(sc_um, axis=1), 's-', label='SC', color='darkorange')
axes[3].axhline(0, color='k', lw=0.5)
axes[3].legend()
axes[3].set_title('SD profile comparison')
plt.tight_layout()
plt.show()

## 4. Phase Presets

What do the models look like at different learning stages?

In [ ]:
PRESETS = {
    'Naive': {
        'BE': dict(sigma_percep=0.20, A_repulsion=0.05, eta_learning=0.45, eta_relax=0.08),
        'SC': dict(sigma_percep=0.20, A_repulsion=0.05, gamma=0.80, sigma_update=0.4),
    },
    'Expert': {
        'BE': dict(sigma_percep=0.12, A_repulsion=0.10, eta_learning=0.15, eta_relax=0.15),
        'SC': dict(sigma_percep=0.12, A_repulsion=0.10, gamma=0.98, sigma_update=0.3),
    },
    'Post-shift': {
        'BE': dict(sigma_percep=0.15, A_repulsion=0.08, eta_learning=0.40, eta_relax=0.10),
        'SC': dict(sigma_percep=0.15, A_repulsion=0.08, gamma=0.85, sigma_update=0.35),
    },
}

fig, axes = plt.subplots(len(PRESETS), 2, figsize=(10, 4 * len(PRESETS)))
for row, (phase, models) in enumerate(PRESETS.items()):
    be_um, _, be_acc, _ = simulate_be(**models['BE'])
    sc_um, _, sc_acc, _ = simulate_sc(**models['SC'])
    plot_update_matrix(be_um, ax=axes[row, 0],
                       title=f'{phase} BE (acc={be_acc:.2f})')
    plot_update_matrix(sc_um, ax=axes[row, 1],
                       title=f'{phase} SC (acc={sc_acc:.2f})')
plt.tight_layout()
plt.show()